In [6]:
import numpy as np

def forward_algorithm(observation_sequence, transition_matrix, emission_matrix, initial_distribution):
    num_states = len(initial_distribution)
    num_symbols = len(emission_matrix[0])
    sequence_length = len(observation_sequence)

    # Initialize the forward probabilities matrix
    alpha = np.zeros((num_states, sequence_length))

    # Initialization step
    for i in range(num_states):
        alpha[i][0] = initial_distribution[i] * emission_matrix[i][observation_sequence[0]]

    # Recursion step
    for t in range(1, sequence_length):
        for j in range(num_states):
            alpha[j][t] = np.sum(alpha[i][t - 1] * transition_matrix[i][j] * emission_matrix[j][observation_sequence[t]]
                                 for i in range(num_states))

    return alpha

def backward_algorithm(observation_sequence, transition_matrix, emission_matrix):
    num_states = len(transition_matrix)
    sequence_length = len(observation_sequence)

    # Initialize the backward probabilities matrix
    beta = np.zeros((num_states, sequence_length))

    # Initialization step
    for i in range(num_states):
        beta[i][-1] = 1

    # Recursion step
    #البداية من -2 وهروح -1 والفرق بين index -1  
    # -2 هو ان البداية من وهتروح ل -1  وفرق الindex   
    for t in range(sequence_length - 2, -1, -1):
        for i in range(num_states):
            beta[i][t] = np.sum(transition_matrix[i][j] * emission_matrix[j][observation_sequence[t + 1]] * beta[j][t + 1]
                                for j in range(num_states))

    return beta

def baum_welch(observation_sequence, num_states, num_symbols, num_iterations=100):
    # Randomly initialize the model parameters
    transition_matrix = np.random.rand(num_states, num_states)
    emission_matrix = np.random.rand(num_states, num_symbols)
    initial_distribution = np.random.rand(num_states)

    for iteration in range(num_iterations):
        # E-step: Forward-Backward algorithm
        alpha = forward_algorithm(observation_sequence, transition_matrix, emission_matrix, initial_distribution)
        beta = backward_algorithm(observation_sequence, transition_matrix, emission_matrix)

        # M-step: Update parameters
        xi = np.zeros((num_states, num_states, len(observation_sequence) - 1))
        gamma = np.zeros((num_states, len(observation_sequence)))

        for t in range(len(observation_sequence) - 1):
            
            denominator = np.sum(alpha[i][t] * beta[i][t] for i in range(num_states))

            for i in range(num_states):
                gamma[i][t] = (alpha[i][t] * beta[i][t]) / denominator
                #
                for j in range(num_states):
                    xi[i][j][t] = (alpha[i][t] * transition_matrix[i][j] * emission_matrix[j][observation_sequence[t + 1]] * beta[j][t + 1]) / denominator

        # Update initial distribution,transition matrix and emission matrix 
        for i in range(num_states):
            initial_distribution[i] = gamma[i][0]

            for j in range(num_states):
                transition_matrix[i][j] = np.sum(xi[i][j]) / np.sum(gamma[i][:len(observation_sequence) - 1])

        for j in range(num_states):
            for k in range(num_symbols):
                emission_matrix[j][k] = np.sum(gamma[i][t] for t in range(len(observation_sequence) - 1) if observation_sequence[t] == k) / np.sum(gamma[i][:len(observation_sequence) - 1])

    return  initial_distribution, transition_matrix, emission_matrix

# Example usage
observation_sequence = [1, 0, 1, 1, 0, 1, 0, 0, 1]
num_states = 2
num_symbols = 2

result = baum_welch(observation_sequence, num_states, num_symbols)
print("Updated Transition Matrix:\n", result[0])
print("Updated Emission Matrix:\n", result[1])
print("Updated Initial Distribution:\n", result[2])



Updated Transition Matrix:
 [0.07134587 0.92865413]
Updated Emission Matrix:
 [[0.2257402  0.7742598 ]
 [0.05023027 0.94976973]]
Updated Initial Distribution:
 [[0.50059751 0.49940249]
 [0.50059751 0.49940249]]


C:\Users\Blu-Ray\AppData\Local\Temp\ipykernel_17200\1435341838.py:18: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  alpha[j][t] = np.sum(alpha[i][t - 1] * transition_matrix[i][j] * emission_matrix[j][observation_sequence[t]]
C:\Users\Blu-Ray\AppData\Local\Temp\ipykernel_17200\1435341838.py:39: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  beta[i][t] = np.sum(transition_matrix[i][j] * emission_matrix[j][observation_sequence[t + 1]] * beta[j][t + 1]
C:\Users\Blu-Ray\AppData\Local\Temp\ipykernel_17200\1435341838.py:61: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  denominator = np.sum(alpha[

In [7]:
import numpy as np

def forward_algorithm(observation_sequence, transition_matrix, emission_matrix, initial_distribution):
    num_states = len(initial_distribution)
    num_symbols = len(emission_matrix[0])
    sequence_length = len(observation_sequence)

    alpha = np.zeros((num_states, sequence_length))

    for i in range(num_states):
        alpha[i][0] = initial_distribution[i] * emission_matrix[i][observation_sequence[0]]

    for t in range(1, sequence_length):
        for j in range(num_states):
            alpha[j][t] = np.sum(alpha[i][t - 1] * transition_matrix[i][j] * emission_matrix[j][observation_sequence[t]]
                                 for i in range(num_states))

    return alpha

def backward_algorithm(observation_sequence, transition_matrix, emission_matrix):
    num_states = len(transition_matrix)
    sequence_length = len(observation_sequence)

    beta = np.zeros((num_states, sequence_length))

    for i in range(num_states):
        beta[i][-1] = 1

    for t in range(sequence_length - 2, -1, -1):
        for i in range(num_states):
            beta[i][t] = np.sum(transition_matrix[i][j] * emission_matrix[j][observation_sequence[t + 1]] * beta[j][t + 1]
                                for j in range(num_states))

    return beta

def baum_welch(observation_sequence, transition_matrix, emission_matrix, initial_distribution, num_iterations=20):
    num_states, num_symbols = transition_matrix.shape

    for iteration in range(num_iterations):
        alpha = forward_algorithm(observation_sequence, transition_matrix, emission_matrix, initial_distribution)
        beta = backward_algorithm(observation_sequence, transition_matrix, emission_matrix)

        # Print the output of forward_algorithm and backward_algorithm
        print(f"Iteration {iteration + 1}:")
        print("Forward Algorithm Result:\n", alpha)
        print("Backward Algorithm Result:\n", beta)

        xi = np.zeros((num_states, num_states, len(observation_sequence) - 1))
        
        gamma = np.zeros((num_states, len(observation_sequence)))

        for t in range(len(observation_sequence) - 1):
            # Calculate the denominator for normalization
            denominator = np.sum(alpha[i][t] * beta[i][t] for i in range(num_states))
            # Update the gamma matrix, which represents the probability of being in each state at time t
            for i in range(num_states):
                gamma[i][t] = (alpha[i][t] * beta[i][t]) / denominator
               # Update the xi matrix, which represents the joint probability of being in states i and j at times t and t+1
                for j in range(num_states):
                    xi[i][j][t] = (alpha[i][t] * transition_matrix[i][j] * emission_matrix[j][observation_sequence[t + 1]] * beta[j][t + 1]) / denominator
         
        # Update transition matrix, emission matrix, and initial distribution
        for i in range(num_states):
            initial_distribution[i] = gamma[i][0]

            for j in range(num_states):
                transition_matrix[i][j] = np.sum(xi[i][j]) / np.sum(gamma[i][:len(observation_sequence) - 1])

        for j in range(num_states):
            for k in range(num_symbols):
                emission_matrix[j][k] = np.sum(gamma[i][t] for t in range(len(observation_sequence) - 1) if observation_sequence[t] == k) / np.sum(gamma[i][:len(observation_sequence) - 1])

    return transition_matrix, emission_matrix, initial_distribution

# Example usage
observation_sequence = [1, 0, 1]
num_states = 2
num_symbols = 2

# Get user input for matrices
transition_matrix = np.array([[float(input(f"Enter transition probability from state {i} to state {j}: ")) for j in range(num_states)] for i in range(num_states)])
emission_matrix = np.array([[float(input(f"Enter emission probability of symbol {k} from state {j}: ")) for k in range(num_symbols)] for j in range(num_states)])
initial_distribution = np.array([float(input(f"Enter initial distribution probability of state {i}: ")) for i in range(num_states)])

result = baum_welch(observation_sequence, transition_matrix, emission_matrix, initial_distribution)
print("Updated Transition Matrix:\n", result[0])
print("Updated Emission Matrix:\n", result[1])
print("Updated Initial Distribution:\n", result[2])


Enter transition probability from state 0 to state 0: 0.99
Enter transition probability from state 0 to state 1: 0.01
Enter transition probability from state 1 to state 0: 0.01
Enter transition probability from state 1 to state 1: 0.99
Enter emission probability of symbol 0 from state 0: 0.8
Enter emission probability of symbol 1 from state 0: 0.1
Enter emission probability of symbol 0 from state 1: 0.2
Enter emission probability of symbol 1 from state 1: 0.9
Enter initial distribution probability of state 0: 0.99
Enter initial distribution probability of state 1: 0.01
Iteration 1:
Forward Algorithm Result:
 [[0.099     0.07848   0.0077715]
 [0.009     0.00198   0.0024705]]
Backward Algorithm Result:
 [[0.08732 0.108   1.     ]
 [0.17748 0.892   1.     ]]
Iteration 2:
Forward Algorithm Result:
 [[0.40083647 0.19973687 0.09003175]
 [0.07406452 0.04963317 0.02839433]]
Backward Algorithm Result:
 [[0.24937004 0.474901   1.        ]
 [0.24937004 0.474901   1.        ]]
Iteration 3:
Forward

C:\Users\Blu-Ray\AppData\Local\Temp\ipykernel_17200\4203843382.py:15: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  alpha[j][t] = np.sum(alpha[i][t - 1] * transition_matrix[i][j] * emission_matrix[j][observation_sequence[t]]
C:\Users\Blu-Ray\AppData\Local\Temp\ipykernel_17200\4203843382.py:31: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  beta[i][t] = np.sum(transition_matrix[i][j] * emission_matrix[j][observation_sequence[t + 1]] * beta[j][t + 1]
C:\Users\Blu-Ray\AppData\Local\Temp\ipykernel_17200\4203843382.py:52: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  denominator = np.sum(alpha[

In [8]:
import numpy as np

def forward_algorithm(observation_sequence, transition_matrix, emission_matrix, initial_distribution):
    num_states = len(initial_distribution)
    num_symbols = len(emission_matrix[0])
    sequence_length = len(observation_sequence)

    alpha = np.zeros((num_states, sequence_length))

    for i in range(num_states):
        alpha[i][0] = initial_distribution[i] * emission_matrix[i][observation_sequence[0]]

    for t in range(1, sequence_length):
        for j in range(num_states):
            alpha[j][t] = np.sum(alpha[i][t - 1] * transition_matrix[i][j] * emission_matrix[j][observation_sequence[t]]
                                 for i in range(num_states))

    return alpha

def backward_algorithm(observation_sequence, transition_matrix, emission_matrix):
    num_states = len(transition_matrix)
    sequence_length = len(observation_sequence)

    beta = np.zeros((num_states, sequence_length))

    for i in range(num_states):
        beta[i][-1] = 1

    for t in range(sequence_length - 2, -1, -1):
        for i in range(num_states):
            beta[i][t] = np.sum(transition_matrix[i][j] * emission_matrix[j][observation_sequence[t + 1]] * beta[j][t + 1]
                                for j in range(num_states))

    return beta

def baum_welch(observation_sequence, num_states, num_symbols, num_iterations=5):
    # Get user input for initial values
    initial_transition_matrix = np.array([[float(input(f"Enter transition probability from state {i} to state {j}: ")) for j in range(num_states)] for i in range(num_states)])
    initial_emission_matrix = np.array([[float(input(f"Enter emission probability of symbol {k} from state {j}: ")) for k in range(num_symbols)] for j in range(num_states)])
    initial_initial_distribution = np.array([float(input(f"Enter initial distribution probability of state {i}: ")) for i in range(num_states)])

    forward_matrices = []
    backward_matrices = []

    for iteration in range(num_iterations):
        # E-step: Forward-Backward algorithm
        alpha = forward_algorithm(observation_sequence, initial_transition_matrix, initial_emission_matrix, initial_initial_distribution)
        beta = backward_algorithm(observation_sequence, initial_transition_matrix, initial_emission_matrix)

        forward_matrices.append(alpha)
        backward_matrices.append(beta)

        # M-step: Update parameters
        xi = np.zeros((num_states, num_states, len(observation_sequence) - 1))
        gamma = np.zeros((num_states, len(observation_sequence)))

        for t in range(len(observation_sequence) - 1):
            # Calculate the denominator for normalization
            denominator = np.sum(alpha[i][t] * beta[i][t] for i in range(num_states))

            # Update the gamma matrix, which represents the probability of being in each state at time t
            for i in range(num_states):
                gamma[i][t] = (alpha[i][t] * beta[i][t]) / denominator
                
                # Update the xi matrix, which represents the joint probability of being in states i and j at times t and t+1
                for j in range(num_states):
                    xi[i][j][t] = (alpha[i][t] * initial_transition_matrix[i][j] * initial_emission_matrix[j][observation_sequence[t + 1]] * beta[j][t + 1]) / denominator

        # Update transition matrix, emission matrix, and initial distribution
        for i in range(num_states):
            initial_initial_distribution[i] = gamma[i][0]

            for j in range(num_states):
                initial_transition_matrix[i][j] = np.sum(xi[i][j]) / np.sum(gamma[i][:len(observation_sequence) - 1])

        for j in range(num_states):
            for k in range(num_symbols):
                initial_emission_matrix[j][k] = np.sum(gamma[i][t] for t in range(len(observation_sequence) - 1) if observation_sequence[t] == k) / np.sum(gamma[i][:len(observation_sequence) - 1])

    return initial_transition_matrix, initial_emission_matrix, initial_initial_distribution, forward_matrices, backward_matrices

# Example usage
observation_sequence = [1, 0, 1]
num_states = 2
num_symbols = 2

result = baum_welch(observation_sequence, num_states, num_symbols)
print("Updated Transition Matrix:\n", result[0])
print("Updated Emission Matrix:\n", result[1])
print("Updated Initial Distribution:\n", result[2])

# Print forward and backward matrices for each iteration
for i in range(len(result[3])):
    print(f"Iteration {i + 1}:")
    print("Forward Algorithm Result:\n", result[3][i])
    print("Backward Algorithm Result:\n", result[4][i])


Enter transition probability from state 0 to state 0: 0.99
Enter transition probability from state 0 to state 1: 0.01
Enter transition probability from state 1 to state 0: 0.01
Enter transition probability from state 1 to state 1: 0.99
Enter emission probability of symbol 0 from state 0: 0.8
Enter emission probability of symbol 1 from state 0: 0.1
Enter emission probability of symbol 0 from state 1: 0.2
Enter emission probability of symbol 1 from state 1: 0.9
Enter initial distribution probability of state 0: 0.99
Enter initial distribution probability of state 1: 0.01
Updated Transition Matrix:
 [[0.9484282  0.0515718 ]
 [0.00290057 0.99709943]]
Updated Emission Matrix:
 [[0.56067235 0.43932765]
 [0.56067235 0.43932765]]
Updated Initial Distribution:
 [0.84404218 0.15595782]
Iteration 1:
Forward Algorithm Result:
 [[0.099     0.07848   0.0077715]
 [0.009     0.00198   0.0024705]]
Backward Algorithm Result:
 [[0.08732 0.108   1.     ]
 [0.17748 0.892   1.     ]]
Iteration 2:
Forward Al

C:\Users\Blu-Ray\AppData\Local\Temp\ipykernel_17200\704995550.py:15: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  alpha[j][t] = np.sum(alpha[i][t - 1] * transition_matrix[i][j] * emission_matrix[j][observation_sequence[t]]
C:\Users\Blu-Ray\AppData\Local\Temp\ipykernel_17200\704995550.py:31: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  beta[i][t] = np.sum(transition_matrix[i][j] * emission_matrix[j][observation_sequence[t + 1]] * beta[j][t + 1]
C:\Users\Blu-Ray\AppData\Local\Temp\ipykernel_17200\704995550.py:59: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  denominator = np.sum(alpha[i][